# Convert qs file into h5ad

In [1]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.sparse import csc_matrix

import rpy2.robjects as ro
r = ro.r

# Folder to save tmp files
#QS_PATH = "/home/gdallagl/myworkdir/XDP/data/XDP/artificial_bican/geneset_001/zonated_objs_combined_with_md__combined__rep_001__ventral_matrix_keep_1.0.qs"
GENESET="004"
QS_PATH = f"/home/gdallagl/myworkdir/XDP/data/XDP/artificial_bican/geneset_{GENESET}/initial_perturbed/zonated_objs_combined_with_md__combined__rep_001__ventral_matrix_keep_1.0.qs"
base = os.path.splitext(QS_PATH)[0]
os.path.exists(QS_PATH)

# counts_mtx    = base + "_counts.mtx"
# genes_tsv     = base + "_genes.tsv"
# barcodes_tsv  = base + "_barcodes.tsv"
# metadata_csv  = base + "_metadata.csv"

True

In [2]:
r(f'''

    library(qs)
    library(Seurat)

    # Set the path
    QS_PATH <- "{QS_PATH}"
    QS_PATH_NOEXT <- tools::file_path_sans_ext(QS_PATH)

    # Load the Seurat object
    print("Loading Seurat object...")
    seurat <- qread(QS_PATH)
    print(seurat)
    print(paste0("Loaded: ", ncol(seurat), " cells, ", nrow(seurat), " genes"))

    # Extract the counts matrix
    counts <- GetAssayData(seurat, assay = "RNA", layer = "counts")
    summary(counts)

    # Save files
    print("Saving counts matrix...")
    saveRDS(counts, paste0(QS_PATH_NOEXT, "_counts.rds"), compress = FALSE)

    print("Saving gene names...")
    write.table(rownames(counts), paste0(QS_PATH_NOEXT, "_genes.tsv"), 
                quote = FALSE, row.names = FALSE, col.names = FALSE)

    print("Saving cell barcodes...")
    write.table(colnames(counts), paste0(QS_PATH_NOEXT, "_barcodes.tsv"), 
                quote = FALSE, row.names = FALSE, col.names = FALSE)

    print("Saving metadata...")
    write.csv(seurat@meta.data, paste0(QS_PATH_NOEXT, "_metadata.csv"), row.names = TRUE)

''')

R callback write-console: qs 0.27.3. Announcement: https://github.com/qsbase/qs/issues/103
  
R callback write-console: Loading required package: SeuratObject
  
R callback write-console: Loading required package: sp
  


R callback write-console: 
Attaching package: ‘SeuratObject’

  
R callback write-console: The following objects are masked from ‘package:base’:

    %||%, intersect, t

  



    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    

R callback write-console: 
Attaching package: ‘Seurat’

  
R callback write-console: The following object is masked from ‘package:base’:

    %||%

  


[1] "Loading Seurat object..."
An object of class Seurat 
37905 features across 191639 samples within 1 assay 
Active assay: RNA (37905 features, 0 variable features)
 1 layer present: counts
[1] "Loaded: 191639 cells, 37905 genes"
[1] "Saving counts matrix..."
[1] "Saving gene names..."
[1] "Saving cell barcodes..."
[1] "Saving metadata..."


In [3]:
print("\nLoading sparse matrix from RDS...")

# Load the RDS file
counts_r = r.readRDS(base + "_counts.rds")

# Extract sparse matrix components manually
# R's dgCMatrix has slots: x (values), i (row indices), p (column pointers), Dim (dimensions)
data = np.array(counts_r.slots['x'])
indices = np.array(counts_r.slots['i'])
indptr = np.array(counts_r.slots['p'])
shape = tuple(counts_r.slots['Dim'])

# Create scipy sparse matrix and transpose for scanpy
X = csc_matrix((data, indices, indptr), shape=shape).T.tocsr() # TRANSPOSE

# Load gene and cell names
genes = pd.read_csv(base + "_genes.tsv", header=None)[0].values
cells = pd.read_csv(base + "_barcodes.tsv", header=None)[0].values

# Create AnnData object
adata = sc.AnnData(X=X)
adata.var_names = genes
adata.obs_names = cells

# Load and attach metadata
metadata = pd.read_csv(base + "_metadata.csv", index_col=0)
adata.obs = metadata.loc[adata.obs_names]

# Save as h5ad
adata.write_h5ad(base + "_raw.h5ad")

print(f"\nSuccess! Created AnnData with {adata.n_obs} cells and {adata.n_vars} genes")
print(f"Saved to: {base}_raw.h5ad")



Loading sparse matrix from RDS...


/tmp/ipykernel_373170/1878117026.py:26: DtypeWarning: Columns (39,49) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(base + "_metadata.csv", index_col=0)



Success! Created AnnData with 191639 cells and 37905 genes
Saved to: /home/gdallagl/myworkdir/XDP/data/XDP/artificial_bican/geneset_004/initial_perturbed/zonated_objs_combined_with_md__combined__rep_001__ventral_matrix_keep_1.0_raw.h5ad


In [4]:
adata

AnnData object with n_obs × n_vars = 191639 × 37905
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'Row.names', 'donor_id', 'PREFIX', 'CELL_BARCODE', 'NUM_GENIC_READS', 'NUM_TRANSCRIPTS', 'NUM_GENES', 'num_retained_transcripts', 'pct_coding', 'pct_utr', 'pct_intergenic', 'pct_genic', 'pct_intronic', 'pct_mt', 'pct_ribosomal', 'frac_contamination', 'experiment', 'Neighborhood_name', 'Neighborhood_bootstrapping_probability', 'Class_name', 'Class_bootstrapping_probability', 'Subclass_name', 'Subclass_bootstrapping_probability', 'Group_name', 'Group_bootstrapping_probability', 'Cluster_name', 'Cluster_alias', 'Cluster_bootstrapping_probability', 'x', 'y', 'library', 'donor', 'x_um2', 'y_um2', 'unique_cell_ID', 'cb', 'Final_Zone_Assignments', 'Group_name.1', 'cell_type', 'Cohort', 'Biobank', 'Age.at.Death', 'Sex', 'Race', 'Ethnicity', 'Cause.of.Death', 'PMI', 'spn_type', 'case_control', 'RNA_snn_res.0.2', 'seurat_clusters', 'RNA_snn_res.0.5', 'PC_1', 'PC_2', 'PC_3', 'PC_4', 'PC_5', 'P

In [5]:
import numpy as np

# all integers?
np.all(np.mod(adata.X.data, 1) == 0)


np.True_

In [6]:
# After loading the matrix
print(f"Matrix shape: {X.shape}")
print(f"Number of genes: {len(genes)}")
print(f"Number of cells: {len(cells)}")
assert X.shape == (len(cells), len(genes)), "Dimension mismatch!"

# Before assigning metadata
print(f"Cells in adata: {len(adata.obs_names)}")
print(f"Cells in metadata: {len(metadata)}")
assert all(cell in metadata.index for cell in adata.obs_names), "Missing cells in metadata!"

Matrix shape: (191639, 37905)
Number of genes: 37905
Number of cells: 191639
Cells in adata: 191639
Cells in metadata: 191639


In [7]:
obs_df = adata.obs.copy()

for col in obs_df.columns:
    unique_vals = obs_df[col].unique()
    print(f"Column: {col}")
    print(f"Unique values ({len(unique_vals)}): {unique_vals}\n")

Column: orig.ident
Unique values (15): ['2024-10-25', '2025-03-11', '2025-03-18', '2025-03-19', '2025-03-24', ..., '2025-04-15', '2025-05-13', '2025-05-23', '2025-05-29', '2025-06-04']
Length: 15
Categories (15, object): ['2024-10-25', '2025-03-11', '2025-03-18', '2025-03-19', ..., '2025-05-29', '2025-05-30', '2025-06-03', '2025-06-04']

Column: nCount_RNA
Unique values (103953): [149923. 105479.  86852. ...  82067.  49818.  17312.]



Column: nFeature_RNA
Unique values (10947): [11741 10583  9882 ...  2762 13358 13482]

Column: Row.names
Unique values (191639): ['STR_D1_Matrix_MSN_s5_s5_AACAGGAGTGCACGTA_1'
 'STR_D1_Matrix_MSN_s5_s5_ACATTGGTCGGGTATC_1'
 'STR_D1_Matrix_MSN_s5_s5_ATTGCATAGACCAATT_1' ...
 's38_2025-06-04_s38_GEX_CAP_rxn4_CGTTTCAGTCTGTATT'
 's38_2025-06-04_s38_GEX_CAP_rxn4_ACGTCCGAGCTGGCAT'
 's38_2025-06-04_s38_GEX_CAP_rxn4_CGGTCGTCACGCAATT']

Column: donor_id
Unique values (19): ['PT13935', 'UMBEB23073', 'UMBEB23033', 'MS913848', 'UMBEB23127', ..., 'UMBEB24013', 'UMBEB23158', 'MD6927', 'MS876075', 'MS986638']
Length: 19
Categories (19, object): ['MD6927', 'MD9129', 'MD9162', 'MD9244', ..., 'UMBEB23127', 'UMBEB23158', 'UMBEB23164', 'UMBEB24013']

Column: PREFIX
Unique values (77): ['2024-10-25_s5_Slide-tag_10X-GEMX-5P-GEX_BN_rxn1', '2024-10-25_s5_Slide-tag_10X-GEMX-5P-GEX_BN_rxn2', '2024-10-25_s5_Slide-tag_10X-GEMX-5P-GEX_BN_rxn3', '2024-10-25_s5_Slide-tag_10X-GEMX-5P-GEX_BN_rxn4', '2024-10-25_s5_Slide-t

In [8]:
# adata.obs.donor_id.unique()
# adata.obs.region.unique()
# adata.obs.repeat_length.unique()
# adata.obs.age_of_onset.unique()
# adata.obs.age_of_death.unique()
# adata.obs.disease_duration.unique()
# adata.obs["immediate.cause.of.death"].unique()
# adata.obs.infection_related_death.unique()
# adata.obs.sex.unique()
# adata.obs.condition.unique()


# adata.obs.columns



# table = (
#     adata.obs
#     .groupby(["infection_related_death", "condition"])["donor_id"]
#     .nunique()
#     .unstack(fill_value=0) 
# )

# table